# Tests: `fastermodels.measurement` (source `nbs/04_measurement.ipynb`)

In [ ]:
from fastcore.test import *
import json, tempfile
from pathlib import Path

import numpy as np

import fastermodels
from fastermodels import measurement

In [ ]:
_ROW = dict(schema_version=1, run_id='r1', measured_at='2026-09-28T10:00:00Z', series='2026-09-28',
            harness_commit='deadbee', fasterai_version='0.4.0', report_url=None, hf_repo=None,
            model_id='org/resnet50-p30', model_revision='a1b2c3d', task='classification',
            architecture='resnet50', base_weights='timm/resnet50.a1_in1k', weights_license='apache-2.0',
            variant='p30-int8', recipe={'prune': 0.30, 'quant': 'int8', 'calib_n': 512}, input_resolution=224,
            macs=2_900_000_000, params=17_000_000, model_hash=None, engine_hash='ab' * 32, size_mb=18.2,
            device='jetson-nano-4gb', device_host='own', runtime='tensorrt', runtime_version='8.2.1',
            platform_version='jetpack-4.6.1', precision='int8', int8_mode='explicit', power_mode='MAXN',
            clocks_locked=True, batch_size=4,
            temp_start_c=41.0, temp_end_c=47.5, gpu_freq_mhz=921.6, throttled=False, warmup_s=10.0,
            duration_s=60.0, repeats=5, latency_spread_pct=1.2,
            latency_ms_p50=8.0, latency_ms_p90=8.4, latency_ms_p99=9.1, latency_scope='engine execute',
            host_latency_ms=8.5, latency_source='measured',
            metric='top1', eval_set='imagenet-val', eval_set_hash='sha256:0f', dataset_license='custom',
            eval_n=50_000, accuracy=78.9, ci_low=78.5, ci_high=79.3, ci_method='wilson',
            measured_through='deployed_engine', argmax_parity=99.7, parity_floor=99.0, baseline_run_id='r0')


def problems(**change):
    "measurement.validate() on the valid row with some fields changed; a value of `...` removes the field"
    r = {**_ROW, **change}
    return measurement.validate({k: v for k, v in r.items() if v is not ...})


def names(ps): return {p.split(':')[0] for p in ps}

In [ ]:
test_eq(set(_ROW), set(measurement.FIELDS))
test_eq(measurement.validate(_ROW), [])
test_eq(measurement.SCHEMA_VERSION, 1)
# optional fields may be absent: absent means not recorded
test_eq(problems(series=..., fasterai_version=..., platform_version=..., temp_start_c=..., latency_ms_p90=...,
                 latency_ms_p99=..., baseline_run_id=..., model_hash=...), [])
# JSON writes 8 as 8: an int is accepted where a float is expected
test_eq(problems(latency_ms_p50=8), [])
# numpy scalars, as a harness produces them, are numbers; a numpy or Python bool is not
test_eq(problems(batch_size=np.int64(4), eval_n=np.int32(50_000), macs=np.uint64(2_900_000_000)), [])
test_eq(problems(latency_ms_p50=np.float64(8.0), accuracy=np.float32(78.9)), [])
test_eq(names(problems(batch_size=np.bool_(True))), {'batch_size'})
test_eq(names(problems(latency_ms_p50=np.float64('nan'))), {'latency_ms_p50'})
# the module is reached by its name, its generic function names are not at the package top level
test_eq(fastermodels.measurement is measurement, True)
test_eq(hasattr(fastermodels, 'status') or hasattr(fastermodels, 'key'), False)

In [ ]:
# missing required, by name
test_eq(problems(run_id=...), ['run_id: missing'])
test_eq(problems(latency_ms_p50=None), ['latency_ms_p50: missing'])
test_eq(problems(weights_license=...), ['weights_license: missing'])
test_eq(names(problems(model_id=..., device=...)), {'model_id', 'device'})

# wrong type
test_eq(problems(batch_size='4'), ["batch_size: expected int, got '4'"])
test_eq(names(problems(recipe=[0.3])), {'recipe'})
test_eq(names(problems(variant={'prune': 0.3})), {'variant'})
test_eq(names(problems(runtime_version=8.2)), {'runtime_version'})
test_eq(names(problems(throttled='no')), {'throttled'})
test_eq(names(problems(batch_size=4.0)), {'batch_size'})
test_eq(names(problems(latency_ms_p50=float('nan'))), {'latency_ms_p50'})
test_eq(names(problems(latency_ms_p50=float('inf'))), {'latency_ms_p50'})

# a bool is an int in Python, and neither a count nor a measure here
test_eq(problems(batch_size=True), ['batch_size: expected int, got True'])
test_eq(names(problems(schema_version=True)), {'schema_version'})
test_eq(names(problems(latency_ms_p50=True)), {'latency_ms_p50'})
test_eq(names(problems(macs=False)), {'macs'})
test_eq(names(problems(input_resolution=True)), {'input_resolution'})

# out of range
test_eq(names(problems(batch_size=0)), {'batch_size'})
test_eq(names(problems(repeats=0)), {'repeats'})
test_eq(names(problems(input_resolution=0)), {'input_resolution'})
test_eq(names(problems(latency_ms_p50=0.0)), {'latency_ms_p50'})
test_eq(names(problems(latency_ms_p99=0.0)), {'latency_ms_p99'})
test_eq(names(problems(size_mb=-1.0)), {'size_mb'})
test_eq(names(problems(latency_spread_pct=-0.1)), {'latency_spread_pct'})
test_eq(names(problems(warmup_s=-1.0)), {'warmup_s'})
test_eq(names(problems(duration_s=-1.0)), {'duration_s'})
test_eq(problems(warmup_s=0), [])
test_eq(names(problems(host_latency_ms=0)), {'host_latency_ms'})
test_eq(names(problems(eval_n=0)), {'eval_n'})
test_eq(names(problems(argmax_parity=100.5)), {'argmax_parity'})
test_eq(names(problems(accuracy=150.0, ci_high=160.0)), {'accuracy', 'ci_high'})

# closed vocabularies
test_eq(names(problems(precision='bf16', int8_mode=None)), {'precision'})
test_eq(names(problems(task='segmentation')), {'task'})
test_eq(names(problems(ci_method='normal')), {'ci_method'})
test_eq(names(problems(runtime='TensorRT 8.2.1')), {'runtime'})
test_eq(names(problems(device_host='borrowed')), {'device_host'})
test_eq(names(problems(latency_source='guessed')), {'latency_source'})
test_eq(names(problems(measured_through='notebook')), {'measured_through'})

# schema version mismatch
test_eq(problems(schema_version=2), ['schema_version: 2 not one of (1,)'])

# unknown fields: a derived throughput, a verdict, and a name from before the merge
test_eq(problems(throughput_img_s=500.0), ['throughput_img_s: unknown field'])
test_eq(problems(passed=True), ['passed: unknown field'])
test_eq(problems(latency_ms=8.0), ['latency_ms: unknown field'])

# not a row at all
test_eq(measurement.validate([_ROW]), ['row: expected a dict, got list'])
test_eq(measurement.validate(None), ['row: expected a dict, got NoneType'])

In [ ]:
# int8_mode is required iff the precision is int8
test_eq(problems(int8_mode=None), ['int8_mode: missing, required when precision is int8'])
test_eq(problems(int8_mode=...), ['int8_mode: missing, required when precision is int8'])
test_eq(problems(precision='fp16'), ["int8_mode: 'explicit' given for precision 'fp16'"])
test_eq(problems(precision='mixed'), ["int8_mode: 'explicit' given for precision 'mixed'"])
test_eq(problems(precision='fp16', int8_mode=None), [])
test_eq(problems(precision='mixed', int8_mode=None), [])
test_eq(names(problems(int8_mode='dynamic')), {'int8_mode'})

# the interval brackets the accuracy
test_eq(names(problems(accuracy=80.0)), {'ci_low, accuracy, ci_high'})
test_eq(names(problems(ci_low=79.0)), {'ci_low, accuracy, ci_high'})
test_eq(problems(ci_low=78.9, ci_high=78.9), [])

# latency percentiles are ordered, among those recorded
test_eq(names(problems(latency_ms_p90=7.9)), {'latency_ms_p50, latency_ms_p90, latency_ms_p99'})
test_eq(names(problems(latency_ms_p99=8.3)), {'latency_ms_p50, latency_ms_p90, latency_ms_p99'})
test_eq(names(problems(latency_ms_p90=..., latency_ms_p99=7.0)), {'latency_ms_p50, latency_ms_p90, latency_ms_p99'})
test_eq(problems(latency_ms_p90=8.0, latency_ms_p99=8.0), [])
test_eq(problems(latency_ms_p90=None), [])

# the engine digest is 64 lowercase hex characters
test_eq(names(problems(engine_hash='ab' * 31)), {'engine_hash'})
test_eq(names(problems(engine_hash='AB' * 32)), {'engine_hash'})
test_eq(names(problems(engine_hash='sha256:' + 'ab' * 32)), {'engine_hash'})

# the date is ISO-8601 in UTC, and a real date
test_eq(problems(measured_at='2026-09-28T10:00:00.123+00:00'), [])
test_eq(names(problems(measured_at='2026-09-28')), {'measured_at'})
test_eq(names(problems(measured_at='2026-09-28T10:00:00+02:00')), {'measured_at'})
test_eq(names(problems(measured_at='2026-13-28T10:00:00Z')), {'measured_at'})

# recipe keys are free, but the recipe is stored as JSON
test_eq(problems(recipe={}), [])
test_eq(names(problems(recipe={'x': object()})), {'recipe'})

In [ ]:
test_eq(measurement.status(_ROW), ('VALID', []))


def reasons(**change):
    "measurement.status() reasons for the valid row with some fields changed"
    v, rs = measurement.status({k: v for k, v in {**_ROW, **change}.items() if v is not ...})
    test_eq(v, 'INVALID')
    return rs


test_eq(reasons(measured_through='dev_model'), ['measured_through: accuracy not measured through the deployed engine'])
test_eq(reasons(latency_source='estimated'), ['latency_source: latency not measured on the device'])
test_eq(reasons(argmax_parity=98.9), ['argmax_parity: 98.9 below parity_floor 99.0'])
test_eq(reasons(argmax_parity=None), ['argmax_parity: not recorded'])
test_eq(reasons(argmax_parity=...), ['argmax_parity: not recorded'])
test_eq(measurement.status({**_ROW, 'argmax_parity': 99.0}), ('VALID', []))   # at the floor holds
test_eq(reasons(throttled=True), ['throttled: the device throttled during the session'])
test_eq(reasons(throttled=None), ['throttled: not recorded'])
test_eq(reasons(latency_spread_pct=5.1), ['latency_spread_pct: 5.1 above 5.0'])
test_eq(reasons(latency_spread_pct=None), ['latency_spread_pct: not recorded'])
test_eq(measurement.status({**_ROW, 'latency_spread_pct': 5.0}), ('VALID', []))
test_eq(measurement.status({**_ROW, 'latency_spread_pct': 5.1}, max_spread_pct=6.0), ('VALID', []))

# the informational conditions do not decide validity
test_eq(measurement.status({**_ROW, 'temp_start_c': None, 'temp_end_c': None, 'gpu_freq_mhz': None,
                            'clocks_locked': None, 'warmup_s': None}), ('VALID', []))

# every reason is reported, not only the first
test_eq(names(reasons(throttled=None, latency_spread_pct=None, argmax_parity=None, latency_source='estimated',
                      measured_through='dev_model')),
        {'throttled', 'latency_spread_pct', 'argmax_parity', 'latency_source', 'measured_through'})

# a malformed row is invalid for its structural problems
test_eq(reasons(batch_size=True), ['batch_size: expected int, got True'])

In [ ]:
test_eq(measurement.key(_ROW), measurement.key(dict(_ROW)))
# the recipe is canonical: key order does not make a new row
test_eq(measurement.key({**_ROW, 'recipe': {'calib_n': 512, 'quant': 'int8', 'prune': 0.30}}), measurement.key(_ROW))
# a software update on the device is a new key
test_ne(measurement.key({**_ROW, 'runtime_version': '8.5.2'}), measurement.key(_ROW))
test_ne(measurement.key({**_ROW, 'platform_version': 'jetpack-5.1.3'}), measurement.key(_ROW))
for _f, _v in [('batch_size', 1), ('recipe', {'prune': 0.40}), ('variant', 'p40-int8'), ('input_resolution', 160),
               ('device', 'raspberry-pi-5'), ('runtime', 'onnxruntime'), ('measured_at', '2026-09-29T10:00:00Z'),
               ('model_revision', 'e4f5a6b')]:
    test_ne(measurement.key({**_ROW, _f: _v}), measurement.key(_ROW))
# fields outside the identity do not
test_eq(measurement.key({**_ROW, 'latency_ms_p50': 9.0, 'run_id': 'r2', 'series': None}), measurement.key(_ROW))

# throughput is derived: batch_size * 1000 / latency_ms_p50
test_eq(measurement.throughput(_ROW), 500.0)
test_close(measurement.throughput({**_ROW, 'batch_size': 1, 'latency_ms_p50': 2.1}), 476.19, eps=0.01)

In [ ]:
with tempfile.TemporaryDirectory() as _d:
    _p = Path(_d, 'rows.jsonl')

    # a valid row is appended, and reads back equal
    measurement.write_row(_p, _ROW)
    test_eq(measurement.read_rows(_p), [_ROW])
    _r2 = {**_ROW, 'runtime_version': '8.5.2', 'run_id': 'r2'}
    measurement.write_row(str(_p), _r2)
    test_eq(measurement.read_rows(_p), [_ROW, _r2])
    test_eq(len(_p.read_text().splitlines()), 2)

    # an invalid row is refused, naming its problems, and nothing is written
    with ExceptionExpected(ValueError, 'batch_size'): measurement.write_row(_p, {**_ROW, 'batch_size': 0, 'run_id': 'r3'})
    with ExceptionExpected(ValueError, 'throughput_img_s'): measurement.write_row(_p, {**_ROW, 'throughput_img_s': 1.0})
    # a duplicate key is refused, even with other fields changed
    with ExceptionExpected(ValueError, 'already in'): measurement.write_row(_p, {**_ROW, 'run_id': 'again', 'latency_ms_p50': 7.0})
    test_eq(measurement.read_rows(_p), [_ROW, _r2])

    # numpy scalars are written as plain JSON numbers
    _np = {**_ROW, 'run_id': 'r4', 'batch_size': np.int64(2), 'latency_ms_p50': np.float64(4.0),
           'recipe': {'prune': np.float64(0.3), 'calib_n': np.int64(512)}}
    measurement.write_row(_p, _np)
    test_eq(json.loads(_p.read_text().splitlines()[-1])['batch_size'], 2)
    test_eq(measurement.read_rows(_p)[-1]['recipe'], {'prune': 0.3, 'calib_n': 512})
    _p.write_text(''.join(l + '\n' for l in _p.read_text().splitlines()[:-1]))
    test_eq(measurement.read_rows(_p), [_ROW, _r2])

    # a row written by hand without a final newline does not swallow the next one
    _p.write_text(_p.read_text().rstrip('\n'))
    _r3 = {**_ROW, 'batch_size': 1}
    measurement.write_row(_p, _r3)
    test_eq(measurement.read_rows(_p), [_ROW, _r2, _r3])

    # a malformed line is reported with its line number; blank lines are skipped
    _bad = Path(_d, 'bad.jsonl')
    _bad.write_text(json.dumps(_ROW) + '\n\n{"run_id": \n')
    with ExceptionExpected(ValueError, 'bad.jsonl:3'): measurement.read_rows(_bad)
    with ExceptionExpected(ValueError, 'bad.jsonl:3'): measurement.write_row(_bad, _r2)
    _bad.write_text('[1, 2]\n')
    with ExceptionExpected(ValueError, 'bad.jsonl:1: not a JSON object'): measurement.read_rows(_bad)

    # reading a file that is not there is an error, not an empty table
    with ExceptionExpected(FileNotFoundError): measurement.read_rows(Path(_d, 'none.jsonl'))

In [ ]:
import importlib.util, sys

# without the extra installed, a one-line error names it (pandas blocked before any import of it)
_saved = sys.modules.get('pandas')
sys.modules['pandas'] = None
try:
    with ExceptionExpected(ImportError, r"fastermodels\[parquet\]"): measurement.to_parquet([_ROW], 'x.parquet')
finally:
    if _saved is None: sys.modules.pop('pandas', None)
    else: sys.modules['pandas'] = _saved

if importlib.util.find_spec('pyarrow') and importlib.util.find_spec('pandas'):
    import pandas as pd
    with tempfile.TemporaryDirectory() as _d:
        _pq = Path(_d, 'rows.parquet')
        _rows = [_ROW, {**_ROW, 'run_id': 'r2', 'precision': 'fp16', 'int8_mode': None,
                        'recipe': {'quant': 'none', 'prune': 0.30}}]
        _sparse = {k: v for k, v in _ROW.items() if k not in ('series', 'latency_ms_p99')}
        measurement.to_parquet(_rows + [{**_sparse, 'run_id': 'r3'}], _pq)
        _df = pd.read_parquet(_pq)
        # one column per field, in schema order, one line per row
        test_eq(list(_df.columns), list(measurement.FIELDS))
        test_eq(len(_df), 3)
        # the recipe is canonical JSON: key order is not a difference
        test_eq(_df['recipe'][0], '{"calib_n":512,"prune":0.3,"quant":"int8"}')
        test_eq(json.loads(_df['recipe'][1]), {'prune': 0.30, 'quant': 'none'})
        test_eq(_df['latency_ms_p50'].tolist(), [8.0, 8.0, 8.0])
        test_eq(_df['int8_mode'].tolist(), ['explicit', None, 'explicit'])
        # an absent field is a null, not a hole
        test_eq(_df['series'].isna().tolist(), [False, False, True])
        test_eq(_df['latency_ms_p99'].isna().tolist(), [False, False, True])
        # a malformed row is refused with its index, and nothing is written
        with ExceptionExpected(ValueError, 'row 1 refused'): measurement.to_parquet([_ROW, {**_ROW, 'batch_size': 0}], Path(_d, 'no.parquet'))
        test_eq(Path(_d, 'no.parquet').exists(), False)
else: print('pandas or pyarrow not installed: to_parquet round trip skipped')